# Data11 frequency range: supplementary diagnostics

This optional notebook contains strict comparison and accelerometer-reference plots, historical previews, reference timestamp-gap audits and the whole-sweep FFT comparison. The first code cell explicitly selects the completed main tracking run used for the current figures. To use another run from [data11_frequency_range_analysis.ipynb](data11_frequency_range_analysis.ipynb), replace `TRACKING_RUN` with its exact tracking `OUTPUT_PATH`. No latest run is selected automatically. Figure-only `data11_tracking_presentation` exports are not complete tracking inputs.

The parent run supplies the selected records, methods and numerical settings. Historical previews are fixed archived evidence; diagnostic curves remain separate from strict tracking results. Original data and the parent run are read-only. Each new diagnostic analysis creates its own output directory.

For the command-line runner, use `python run_notebooks.py --notebook data11_frequency_range_supplement --tracking-run "<exact tracking output path>"`. The six-notebook `--all` command runs the main analyses only.


In [ ]:
# Select the completed tracking run explicitly.
from pathlib import Path
import importlib
import json
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "release_manifest.json").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
from portable_runtime import project_root, ensure_environment, notebook_path, validate_tracking_run
PROJECT = project_root()
ensure_environment()
NOTEBOOK_PATH = notebook_path("data11_frequency_range_supplement")

# Completed main tracking run used for the current figures; replace for another analysis.
TRACKING_RUN = PROJECT / "results/2026-10-17_data11_harmonic_tracking_run004"
# The CLI passes --tracking-run through IM_TRACKING_RUN to this fresh kernel.
OUTPUT_PATH = validate_tracking_run(os.environ.get("IM_TRACKING_RUN") or TRACKING_RUN)
CONFIG = json.loads((OUTPUT_PATH / "parameters.json").read_text(encoding="utf-8"))["config"]
# Display one selected IM FFT window; this does not change the analysis selection.
RAW_FILE = next((f"IM_{i}.csv" for _, ids in CONFIG["materials"].values() for i in ids), "")
import paper_figure_style as _paper_style
importlib.reload(_paper_style)
print("Read-only tracking source:", OUTPUT_PATH)
display(pd.DataFrame([{"material": name, "noise_ids": ids[0], "sweep_ids": ids[1]}
                      for name, ids in CONFIG["materials"].items()]))


## Strict comparison and accelerometer-reference plots

These figures preserve the strict reference-quality rules. Missing reference points remain blank. The main notebook shows connected nominal-responsivity and accelerometer-reference displays; their added reference points use already-computed estimates from windows rejected only by the timestamp-gap gate. Other quality gates remain active, and strict CSVs and detectable intervals are unchanged. Per-point display values and flags are saved in the parent run's `presentation/` folder.


In [ ]:
# Show the strict versions saved by the explicitly selected tracking run.
_plot_policy = json.loads((OUTPUT_PATH / 'parameters.json').read_text(encoding='utf-8')).get('plot_presentation', {})
for _group, _members in CONFIG['comparison_groups'].items():
    if not any(_material in CONFIG['materials'] for _material in _members):
        continue
    _strict_path = OUTPUT_PATH / f'tracking_comparison_{_group}_strict.png'
    if not _strict_path.is_file() and not _plot_policy:
        # Older tracking runs saved strict figures under the original main filenames.
        _strict_path = OUTPUT_PATH / f'tracking_comparison_{_group}.png'
    if not _strict_path.is_file():
        raise FileNotFoundError(f'Strict comparison figure is missing: {_strict_path}')
    display(Image(filename=str(_strict_path)))

# The independent reference also has a strict version. Earlier connected
# responsivity runs still saved a strict accelerometer figure under this name.
_strict_reference_path = OUTPUT_PATH / 'tracking_accelerometer_reference_strict.png'
if not _strict_reference_path.is_file() and 'main_accelerometer_reference' not in _plot_policy:
    _strict_reference_path = OUTPUT_PATH / 'tracking_accelerometer_reference.png'
if not _strict_reference_path.is_file():
    raise FileNotFoundError(f'Strict accelerometer reference figure is missing: {_strict_reference_path}')
display(Image(filename=str(_strict_reference_path)))


## Historical diagnostic preview

These bundled tables and images are historical results; they do not respond to current settings. They preserve shaded/unshaded curves and linear/logarithmic trajectory diagnostics, including a deferred recording. Displaying them does not reanalyze that recording.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
SAVED_REVIEW = PROJECT / 'historical_display' / '2026-10-13_data11_candidate_review_run003'
SAVED_TRACKING = PROJECT / 'historical_display' / '2026-10-13_data11_selected_tracking_run001'
SAVED_MODEL_CHECK = PROJECT / 'historical_display' / '2026-10-13_data11_sweep_model_check_run001'
for folder in (SAVED_REVIEW, SAVED_TRACKING, SAVED_MODEL_CHECK):
    assert json.loads((folder/'run_status.json').read_text(encoding='utf-8'))['status'] == 'completed'
    print(folder)
display(pd.read_csv(SAVED_REVIEW/'sweep_selection.csv'))
for variant in ('with_region','no_region'):
    display(Image(filename=str(SAVED_TRACKING/f'tracking_fundamental_materials_{variant}.png')))
models = pd.read_csv(SAVED_MODEL_CHECK/'sweep_model_comparison.csv')
display(models[['file','preferred_model','linear_median_abs_error_hz','log_median_abs_error_hz',
                'linear_slope_hz_per_s','linear_30_to_300_duration_s']])
for file in ('IM_26','IM_62'):
    display(Image(filename=str(SAVED_MODEL_CHECK/f'{file}_linear_vs_log.png')))


## Reference timestamp gaps: marked and connected views

Here, gap means an excessive interval between accelerometer timestamps. Both views use the same diagnostic reference estimates at gap-rejected windows. Hollow circles mark them in one version; the other connects them without markers. Other quality gates stay active. Strict results and detectable intervals remain unchanged.


In [ ]:
import data11_reference_gap_plots as reference_gap_plots
importlib.reload(reference_gap_plots)
if not globals().get('OUTPUT_PATH'):
    raise RuntimeError('Select a completed tracking run in the first code cell.')
REFERENCE_GAP_PARENT = Path(OUTPUT_PATH)
REFERENCE_GAP_RESULTS = reference_gap_plots.export_reference_gap_views(
    REFERENCE_GAP_PARENT, NOTEBOOK_PATH)
print('Marked / connected PNG and SVG:', REFERENCE_GAP_RESULTS['output_path'])
for variant in ('marked', 'connected'):
    for figure_path in REFERENCE_GAP_RESULTS['figures']:
        if Path(figure_path).stem == f'tracking_responsivity_materials_reference_gap_{variant}':
            display(Image(filename=figure_path))


## Whole-sweep FFT comparison

This section creates a separate run using the completed tracking run selected at the top of this notebook. It verifies input hashes and roles before using data-fitted trajectory bounds. With $\log f(t)=a+kt$, start and end are the estimated 30 Hz and 300 Hz crossings. Extrapolation beyond trajectory support is recorded. IM and accelerometer are located independently; no synchronization is assumed. Incomplete sweeps remain unavailable.

Sweep, background and reference spectra use direct FFTs. The default boxcar window does not demodulate, track amplitude or search frequency peaks. A Hann option would weight different sweep frequencies differently in time, especially near the ends.


In [ ]:
# whole-sweep-fft: config
from pathlib import Path
import sys, importlib
import numpy as np
import pandas as pd
from IPython.display import display, Image

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "release_manifest.json").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
from portable_runtime import project_root, data_root, ensure_environment, notebook_path
PROJECT = project_root()
ensure_environment()
NOTEBOOK_PATH = notebook_path("data11_frequency_range_supplement")
import data11_whole_sweep_fft as whole_fft
importlib.reload(whole_fft)
if not globals().get('OUTPUT_PATH'):
    raise RuntimeError('Select a completed tracking run in the first code cell.')
FFT_TRACKING_RUN = Path(OUTPUT_PATH)
FFT_CONFIG = whole_fft.fft_config(FFT_TRACKING_RUN)
# File lists and repeat thresholds come from the explicitly selected parent run.
FFT_CONFIG['figure_font_size_pt'] = 17

FFT_CONFIG['fft_window'] = 'boxcar'  # Optional 'hann'; each choice creates a separate run
FFT_CONFIG['fft_pad_factor'] = 1   # No padding; padding does not improve physical resolution
FFT_CONFIG['fft_frequency_step_hz'] = 0.1  # Summary/display grid; native FFT bins are saved separately
FFT_CONFIG['fft_noise_min_windows'] = 1  # Some baselines contain only one complete matched window
FFT_CONFIG['fft_time_overrides_s'] = {}  # Explicit audited per-file bounds only

print('Tracking source:', FFT_TRACKING_RUN)
print('FFT sweep frequency bounds:', FFT_CONFIG['f0_hz'], FFT_CONFIG['f1_hz'])
print('Window:', FFT_CONFIG['fft_window'])


## FFT amplitude, noise and SNR

After strict timestamp checks, data are linearly interpolated to a uniform grid, demeaned, windowed and transformed once:

$$A_{FFT}(f)=2|X(f)|/\sum w,\quad N_{FFT}(f)=\sqrt{\frac1M\sum_j A_{noise,j}(f)^2},\quad SNR_{FFT}(f)=20\log_{10}(A_{FFT}/N_{FFT}).$$

DC and Nyquist are not doubled. Each nonoverlapping noise window matches the sweep's duration, resampled count, window and nfft. Windows begin at the configured noise start; without a complete noise window, SNR is unavailable. Both background conditions are retained. Grid summaries interpolate FFT power before taking its square root; native spectra are saved.

$$\Delta f=1/T,\qquad B_{ENBW}=f_s\sum w^2/(\sum w)^2.$$

A roughly 30.1 s boxcar has ENBW near 0.0332 Hz, distinct from tracking's 1.875 Hz. Whole-sweep FFT-bin amplitude is not instantaneous fundamental amplitude: energy spreads across bins and may include harmonics and leakage. No noise-power subtraction is used; a single background window gives a variable estimate. The 10 dB threshold has no false-alarm guarantee. Shared noise does not add repeats. See [SciPy normalization and ENBW](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.periodogram.html).


## Strict and diagnostic populations

`strict` requires accepted endpoint checks and complete FFT/noise windows without excessive timestamp gaps. `diagnostic_with_gap_interpolation` allows interpolation across gaps while retaining their audit and excluding IM endpoint anomalies. A roughly 30 s FFT window encounters gaps differently from a local 0.8 s tracking window, so accepted samples may differ.

Both populations restrict fundamental conclusions to trajectory-supported frequencies. Spectral energy elsewhere is not evidence of fundamental detectability. Nominal spectral-amplitude responsivity is the ratio of independent condition-mean FFT amplitudes in counts/g, requiring acceptable SNR, repeat counts, duration and ENBW on both sides. It is not calibrated sensitivity, phase or coherence.


In [ ]:
# whole-sweep-fft: run
FFT_RESULTS = None
FFT_OUTPUT_PATH = None
for module_name in ['characterization_run', 'data11_config', 'resolution_sine_fit',
                    'data11_analysis', 'data11_whole_sweep_fft']:
    importlib.reload(importlib.import_module(module_name))
whole_fft = sys.modules['data11_whole_sweep_fft']
shell = get_ipython()
live_cells = list(shell.history_manager.input_hist_raw) if shell is not None else []
FFT_RESULTS = whole_fft.run_whole_fft(
    FFT_CONFIG, NOTEBOOK_PATH, executed_cells=live_cells)
FFT_OUTPUT_PATH = Path(FFT_RESULTS['output_path'])
print('Exact FFT output path:', FFT_OUTPUT_PATH)


In [ ]:
# whole-sweep-fft: window audit
if FFT_RESULTS is None:
    raise RuntimeError('Run the FFT analysis cell first.')
display(FFT_RESULTS['windows'][['file', 'window_start_s', 'window_end_s', 'duration_s',
    'start_extrapolation_s', 'end_extrapolation_s', 'gap_ok', 'gap_interval_count',
    'record_endpoint_count', 'native_df_hz', 'enbw_hz']])
display(FFT_RESULTS['noise_windows'].groupby(['noise_file', 'condition']).agg(
    window_applications=('signal_file', 'size'), gap_free_applications=('gap_ok', 'sum')))
print('Noise window applications may reuse the same baseline across sweeps; not independent repeats.')
print('Input/window issues:', FFT_RESULTS['input_issues'])

def show_fft_figures(*keywords):
    for path in FFT_RESULTS['figures']:
        if any(key in Path(path).name for key in keywords):
            display(Image(filename=str(path)))

show_fft_figures(f'{Path(RAW_FILE).stem}_fft_window',
                 f'accel_{FFT_CONFIG["accel_sweeps"][0]}_fft_window')


## FFT spectra and reference

Plots explicitly label gap-interpolated diagnostics. Strict and diagnostic values are retained separately in `fft_summary.csv` and `fft_responsivity.csv`; diagnostic curves are not strict results.


In [ ]:
# whole-sweep-fft: spectra
show_fft_figures('fft_diagnostic_', 'fft_accelerometer_reference')
display(FFT_RESULTS['reference'].groupby(['population', 'material']).agg(
    eligible_frequency_points=('eligible', 'sum')))


## Compare FFT with harmonic tracking

Each material has amplitude, SNR and reference-normalized panels. Estimators and bandwidths differ. FFT diagnostics permit gap interpolation; tracking retains its main quality gates. `fft_vs_tracking.csv` records whether the samples match at adjacent tracking points. Differences may reflect estimator, bandwidth, full/local windows and accepted samples.


In [ ]:
# whole-sweep-fft: comparison
show_fft_figures('fft_vs_tracking_')
display(FFT_RESULTS['intervals'].groupby(['population', 'material']).agg(
    contiguous_segments=('start_hz', 'size'), first_supported_hz=('start_hz', 'min'),
    last_supported_hz=('end_hz', 'max')))
print('First/last above are an overview, NOT a continuous range when segments > 1.')
print('Every separate interval is saved in fft_detectable_intervals.csv.')
print('Exact FFT output path:', FFT_OUTPUT_PATH)
